# Arm Speed in MolmoAct2

In this notebook, we compare how fast the arms move in each episode, using queries with the [DataFusion DataFrame API](https://datafusion.apache.org/python/).
The joint state holds only joint angles, so we take the largest change of an arm joint from one frame to the next as the arm speed, in radians per frame.
Grippers are left out.
Each episode is summarized by its median arm speed over the frames in which an arm moves.

## Prerequisites

If you have not done so yet, prepare MolmoAct2 RRD files to play with by either (1) downloading the source data and converting it to RRD, or (2) downloading the converted RRD files directly.
Only the base layer is needed.

Then register them in the local catalog on your machine (see the README's [Local Catalog](../README.md#4-local-catalog) section):

```bash
pixi run serve              # in one shell, leave running
pixi run -e molmo register  # in another shell
```

In [ ]:
from __future__ import annotations

import rerun as rr
from datafusion import Expr, col, lit
from datafusion import functions as F
from rerun.notebook import Viewer
from rerun.utilities.datafusion.functions.url_generation import segment_url

SERVER = "rerun+http://127.0.0.1:51234"

In [ ]:
client = rr.catalog.CatalogClient(SERVER)

dataset_name = "molmoact2"
dataset = client.get_dataset(dataset_name)

## Shared setup

In [ ]:
# Set both to limit the search to a range of episodes.
FIRST_EPISODE: int | None = None
LAST_EPISODE: int | None = None

# `observation.state` holds 14 values: left joints 0–5, left gripper, right joints 0–5, right gripper.
ARM_JOINTS = [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12]
# A frame whose arm speed is at most this many radians per frame counts as still and is left out of the median.
MOVING_SPEED = 0.01


# Segment properties show up as length-1 list columns named `property:<group>:<key>`, so `prop()` extracts the scalar.
def prop(name: str) -> Expr:
    """Segment-property columns are length-1 lists; take the scalar."""
    return col(f'"{name}"')[0]


episodes = dataset.segment_table().select(
    "rerun_segment_id",
    prop("property:episode:episode_index").alias("episode"),
    prop("property:episode:instruction").alias("instruction"),
    # The first source file is the episode's data parquet, which holds about ten consecutive episodes.
    prop("property:episode:source_files")[0].alias("data_file"),
)
if FIRST_EPISODE is not None and LAST_EPISODE is not None:
    episodes = episodes.filter(col("episode").between(lit(FIRST_EPISODE), lit(LAST_EPISODE)))
num_episodes = episodes.count()
if num_episodes == 0:
    print("No episodes found!")

print("Episodes selected:", num_episodes)

## Query 1: Median arm speed per episode

One reader over all selected episodes returns `observation.state` per frame.
A window function over each episode's frames takes the difference to the previous frame.

The viewer below lists the episodes from fastest to slowest, with their median arm speed in radians per frame.
Click an episode's `url` to open it in the same viewer, and pick the `Median arm speed` table in the viewer's left panel to go back to the list.

In [ ]:
STATE = col('"/observation.state:Scalars:scalars"')
by_frame = {"partition_by": [col("rerun_segment_id")], "order_by": [col("frame_index").sort()]}

speeds = (
    dataset.filter_segments(episodes)
    .filter_contents(["/observation.state"])
    .reader(index="frame_index")
    .select("rerun_segment_id", "frame_index", STATE.alias("state"))
    .select(
        "rerun_segment_id",
        F.greatest(
            *(F.abs(col("state")[joint] - F.lag(col("state")[joint], **by_frame)) for joint in ARM_JOINTS)
        ).alias("speed"),
    )
)

# Cached, because the table below and Query 2 both read it.
median_speeds = (
    speeds.filter(col("speed") > lit(MOVING_SPEED))
    .aggregate([col("rerun_segment_id")], [F.median(col("speed")).alias("median_speed")])
    .join(episodes.with_column_renamed("rerun_segment_id", "segment"), left_on="rerun_segment_id", right_on="segment")
    .cache()
)

table = (
    median_speeds.with_column("url", segment_url(dataset))
    .select(
        "url",
        "episode",
        F.round(col("median_speed"), lit(4)).alias("median_speed"),
        "instruction",
        "data_file",
    )
    .sort(col("median_speed").sort(ascending=False))
)

viewer = Viewer(width=1200, height=820)
viewer.display()
viewer.send_table("Median arm speed", table)

## Query 2: Median arm speed per data file

Grouping the episodes by their data file shows which runs of consecutive episodes move faster.
The table lists the data files from fastest to slowest.
[observations.md](../observations.md#the-arms-move-faster-in-some-episodes) describes the faster episodes.

In [ ]:
(
    median_speeds.aggregate(
        [col("data_file")],
        [
            F.min(col("episode")).alias("first_episode"),
            F.max(col("episode")).alias("last_episode"),
            F.round(F.min(col("median_speed")), lit(4)).alias("min_speed"),
            F.round(F.median(col("median_speed")), lit(4)).alias("median_speed"),
            F.round(F.max(col("median_speed")), lit(4)).alias("max_speed"),
        ],
    ).sort(col("median_speed").sort(ascending=False))
)